In [1]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

llm = init_chat_model("openai:gpt-5.4-mini")

In [2]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [3]:
@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(
    tools=[get_weather],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)


graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "안녕"},
        ]
    }
)


{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='2a4db0a2-467a-44ec-9518-aa49a472e480'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 125, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DWxNtKnmfHVdDa9n5kHoLKMNz39no', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dae61-d6e5-7a70-9b74-d04ca08b6064-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 14, 'total_tokens': 139, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reaso